# mmCIF Metadata Importer

This notebook provides an **interactive interface** for the metadata importer—no command line or web hosting required. Run cells in order, then:

1. **Upload** your mmCIF file(s)
2. **Select** method-specific and optional specifications
3. **Run** import and download results

Output files are saved in the `notebook_output/` folder (created automatically).

**Protein Data Bank in Europe (PDBe)** · [pdbe.org](https://www.ebi.ac.uk/pdbe)

In [ ]:
import os
import sys
import tempfile
import shutil
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, FileLink, HTML, clear_output

# Ensure we're in the project root (where import_metadata.py and specs/ live)
NOTEBOOK_DIR = Path(os.getcwd())
if (NOTEBOOK_DIR / "import_metadata.py").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
else:
    # Try parent (e.g. if notebook is in a subfolder)
    PROJECT_ROOT = NOTEBOOK_DIR.parent
    if not (PROJECT_ROOT / "import_metadata.py").exists():
        raise FileNotFoundError("Run this notebook from the metadata_import project directory.")

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from import_metadata import import_metadata, detect_method_from_input
import gemmi

OUTPUT_DIR = PROJECT_ROOT / "notebook_output"
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output folder: {OUTPUT_DIR}")

## 1. Upload files & choose options

- **Input mmCIF file** (required): the file to import metadata from.
- **Merge to file** (optional): if provided, metadata is merged into this file instead of creating a new one.
- **Output filename** (optional): only when not merging. Leave empty for `[input_name]_metadata.cif`.
- **Generate log**: create a detailed log of the import.

In [ ]:
upload_input = widgets.FileUpload(
    accept=".cif",
    multiple=False,
    description="Input mmCIF"
)
upload_merge = widgets.FileUpload(
    accept=".cif",
    multiple=False,
    description="Merge to (optional)"
)
output_name = widgets.Text(
    placeholder="e.g. my_metadata.cif (optional)",
    description="Output name:"
)
gen_log = widgets.Checkbox(value=False, description="Generate log file")

display(widgets.VBox([
    widgets.HTML("<b>Input mmCIF file</b> (required)"),
    upload_input,
    widgets.HTML("<b>Merge to file</b> (optional)"),
    upload_merge,
    output_name,
    gen_log
]))

## 2. Select specifications

**Method-specific** (choose at least one): X-ray, X-ray Serial, EM, or NMR. The tool checks that your input file matches the selected method.

**Additional** (optional): Macromolecules, Citation, Authors, Funding, Keywords.

In [ ]:
base = PROJECT_ROOT / "specs"
method_flags = {
    "xray": (str(base / "XRAY.csv"), "X-ray"),
    "xray_serial": (str(base / "XRAY_SERIAL.csv"), "X-ray Serial"),
    "em": (str(base / "EM.csv"), "EM"),
    "nmr": (str(base / "NMR.csv"), "NMR"),
}
optional_flags = {
    "macromolecules": (str(base / "MACROMOLECULES.csv"), "Macromolecules"),
    "citation": (str(base / "CITATION.csv"), "Citation"),
    "authors": (str(base / "AUTHORS.csv"), "Authors"),
    "funding": (str(base / "FUNDING.csv"), "Funding"),
    "keywords": (str(base / "KEYWORDS.csv"), "Keywords"),
}
expected_methods = {
    "xray": ["XRAY"], "xray_serial": ["XRAY"],
    "em": ["EM_MAP_MODEL", "EM_MAP_ONLY", "EM_MODEL_ONLY"],
    "nmr": ["NMR"],
}

method_checks = {k: widgets.Checkbox(description=v[1]) for k, v in method_flags.items()}
optional_checks = {k: widgets.Checkbox(description=v[1]) for k, v in optional_flags.items()}

display(widgets.VBox([
    widgets.HTML("<b>Method-specific</b> (optional; select ≥1 overall)"),
    widgets.HBox([method_checks["xray"], method_checks["xray_serial"], method_checks["em"], method_checks["nmr"]]),
    widgets.HTML("<b>Additional</b> (optional)"),
    widgets.HBox([optional_checks["macromolecules"], optional_checks["citation"], optional_checks["authors"], optional_checks["funding"], optional_checks["keywords"]]),
]))

## 3. Run import

Click **Run import** below. Results appear here and files are written to `notebook_output/`.

In [ ]:
result_out = widgets.Output()

def _save_upload(upload_widget, folder, default_name="uploaded.cif"):
    if not upload_widget.value:
        return None
    info = list(upload_widget.value.values())[0]
    name = info.get("metadata", {}).get("name", default_name)
    path = folder / name
    content = info["content"]
    with open(path, "wb") as f:
        f.write(content if isinstance(content, bytes) else bytes(content))
    return str(path)

def run_extraction(_):
    with result_out:
        clear_output()
        folder = Path(tempfile.mkdtemp())
        try:
            input_path = _save_upload(upload_input, folder, "input.cif")
            if not input_path:
                print("Error: No input file uploaded.")
                return
            merge_path = _save_upload(upload_merge, folder)
            
            spec_files = []
            skipped = []
            
            for k, (spec_path, _) in method_flags.items():
                if not method_checks[k].value:
                    continue
                if not Path(spec_path).exists():
                    print(f"Error: Spec missing: {spec_path}")
                    return
                spec_files.append(spec_path)
            for k, (spec_path, _) in optional_flags.items():
                if optional_checks[k].value:
                    if not Path(spec_path).exists():
                        print(f"Error: Spec missing: {spec_path}")
                        return
                    spec_files.append(spec_path)
            
            if not spec_files:
                print("Error: Select at least one specification (method-specific or additional).")
                return
            
            # Method validation for method-specific specs
            detected = None
            has_method = any(method_checks[k].value for k in method_flags)
            if has_method:
                try:
                    doc = gemmi.cif.read(input_path)
                    detected = detect_method_from_input(doc)
                except Exception as e:
                    print(f"Error detecting method: {e}")
                    return
            
            validated_specs = []
            for k in method_flags:
                if not method_checks[k].value:
                    continue
                spec_path = method_flags[k][0]
                if detected and detected not in expected_methods.get(k, []):
                    reason = f"Input method ({detected}) doesn't match {k}"
                    skipped.append((spec_path, reason))
                    continue
                validated_specs.append(spec_path)
            for k in optional_flags:
                if optional_checks[k].value:
                    validated_specs.append(optional_flags[k][0])
            
            # Output paths
            input_name = Path(input_path).name
            if ".cif.V" in input_name:
                input_stem = input_name.split(".cif.V")[0]
                input_suffix = ".cif.V" + input_name.split(".cif.V")[1]
            else:
                input_stem = input_name[:-4] if input_name.endswith(".cif") else Path(input_name).stem
                input_suffix = ".cif"
            
            merge_output_path = None
            output_path = None
            if merge_path:
                merge_name = Path(merge_path).name
                if ".cif.V" in merge_name:
                    merge_stem = merge_name.split(".cif.V")[0]
                    merge_suffix = ".cif.V" + merge_name.split(".cif.V")[1]
                else:
                    merge_stem = merge_name[:-4] if merge_name.endswith(".cif") else Path(merge_name).stem
                    merge_suffix = ".cif"
                out_name = f"{merge_stem}_merged_with_{input_stem}{merge_suffix}"
                merge_output_path = str(OUTPUT_DIR / out_name)
            else:
                out_name = (output_name.value or "").strip() or f"{input_stem}_metadata.cif"
                if not out_name.endswith(".cif"):
                    out_name += ".cif"
                output_path = str(OUTPUT_DIR / out_name)
            
            log_path = None
            if gen_log.value:
                log_base = Path(merge_output_path or output_path).stem
                log_path = str(OUTPUT_DIR / f"{log_base}.log")
            
            skip_list = skipped if skipped else None
            ok = import_metadata(
                input_path,
                validated_specs,
                output_path,
                merge_to_file=merge_path,
                merge_output_file=merge_output_path,
                log_file=log_path,
                skipped_specs=skip_list,
            )
            if not ok:
                print("Import failed. Check messages above.")
                return
            
            print("\nDone. Output files:")
            for p in [merge_output_path, output_path, log_path]:
                if p and Path(p).exists():
                    rel = Path(p).relative_to(PROJECT_ROOT)
                    display(FileLink(rel))
        finally:
            shutil.rmtree(folder, ignore_errors=True)

btn = widgets.Button(description="Run import")
btn.on_click(run_extraction)
display(btn, result_out)